# Machine Learning-Based BLEVE Peak Pressure Prediction

An event-aware machine learning pipeline for predicting peak pressure from
Boiling Liquid Expanding Vapour Explosion (BLEVE) scenarios.

**Competition results**
- Public leaderboard: **4th place**
- Final leaderboard: **10th out of 448 entries**
- Best public leaderboard MAPE: **0.13211**
- Local validation MAPE: **0.12622**

This notebook documents the complete modelling workflow, from data audit and
event-aware preprocessing through feature engineering, model comparison,
ensemble learning, validation, and final prediction generation.

**Development note:** Generative AI was used for assistance with notebook
organisation and wording. All final code was reviewed, executed, audited, and 
validated by the author.

## 1. Setup

This section imports the libraries and defines the main configuration values used throughout the notebook. The file path constants specify the training file, test file, and final Kaggle prediction file. `TARGET_COL` stores the target variable name, and `EVENT_SIZE = 27` is used because each BLEVE scenario contains up to 27 sensor readings.

The ensemble settings control how the final model is built. `MAIN_SEEDS`, `LOG_SEEDS`, and `HGB_SEEDS` are used to train multiple versions of the neural-network and boosting branches so their predictions can be averaged for stability. `PLAIN_W` and `RESID_W` weight the different MLP architectures inside the Box-Cox and residual branches. `HGB_W` gives a small contribution to the Histogram Gradient Boosting branch. `SIDE_ALPHA` controls how the plain Box-Cox branch and residual branch are blended for each sensor side. `FINAL_CALIB_STRENGTH` controls how strongly the final validation-based calibration is applied.

The `PHYSICS_COLS` list defines the smaller physics-focused feature set used by the Ridge residual branch. These features are selected to represent broad physical trends such as scaled distance, inverse distance, obstacle blockage, temperature ratios, and sensor side effects.


In [ ]:
# Import libraries and define shared constants used throughout the notebook.
import warnings
warnings.filterwarnings("ignore")

from difflib import get_close_matches
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.neural_network import MLPRegressor

from sklearn.preprocessing import ( 
    RobustScaler,
    StandardScaler,
    PowerTransformer, 
)

from sklearn.linear_model import (
    RidgeCV,
    Ridge, 
)

from sklearn.ensemble import (
    HistGradientBoostingRegressor,
    RandomForestRegressor, 
)

from sklearn.metrics import (
    mean_absolute_percentage_error,
    r2_score,
    mean_squared_error,
    mean_absolute_error,
)

# File paths and target/event settings.
# Put the CSV files in the same directory as the notebook 
# Or state the absolute path of the CSV files so the notebook knows where to look for them. 
# On Google Colab, /content/train.csv for example
TRAIN_PATH = "train.csv"            # Change if necessary
TEST_PATH = "test.csv"              # Change if necessary
OUTPUT_PATH = "prediction.csv"      # Change if necessary
TARGET_COL = "Target Pressure (bar)"
EVENT_SIZE = 27

# Ensemble settings: seeds, branch weights, and calibration strength.
FINAL_CALIB_STRENGTH = 1.68
MAIN_SEEDS = [42, 88, 123, 2023, 777, 1001, 2048, 31415]
LOG_SEEDS = [17, 101, 555, 2024, 4096]
HGB_SEEDS = [7, 19, 43]

PLAIN_W = np.array([0.25, 0.65, 0.10], dtype=float)
RESID_W = np.array([0.35, 0.55, 0.10], dtype=float)
HGB_W = 0.10

# Side-specific blend weights for the plain and residual Box-Cox branches.
SIDE_ALPHA = {1.0: 0.73, 2.0: 0.79, 3.0: 0.77, 4.0: 0.73, 5.0: 0.79}
PHYSICS_COLS = [
    "Log_Blast_Scaled_Distance", "Inv_Blast_Scaled_Distance", "Inv_Blast_Scaled_Distance_Sq", "Inv_Blast_Scaled_Distance_Cube", "Obstacle_Blockage",
    "Obstacle_Angle_Sin", "Obstacle_Angle_Cos", "Temp_Ratio", "Vapour_Fraction",
    "Angle_Z", "Abs_Y", "Abs_Z", "Liquid_Ratio_Frac", "Reduced_Pressure",
    "Reduced_Liquid_Temp", "Reduced_Vapour_Temp", "Superheat_Ratio", "Status",
    "Side_1", "Side_2", "Side_3", "Side_4", "Side_5", "Side_1_Inv_Blast_Scaled_Distance", "Side_2_Inv_Blast_Scaled_Distance",
    "Side_3_Inv_Blast_Scaled_Distance", "Side_4_Inv_Blast_Scaled_Distance", "Side_5_Inv_Blast_Scaled_Distance", "Side_1_Obstacle_Blockage",
    "Side_2_Obstacle_Blockage", "Side_3_Obstacle_Blockage", "Side_4_Obstacle_Blockage",
    "Side_5_Obstacle_Blockage",
]

print("Libraries imported.")
print("Train file exists:", Path(TRAIN_PATH).exists())
print("Test file exists:", Path(TEST_PATH).exists())

Libraries imported.
Train file exists: True
Test file exists: True


## 2. Raw Data Audit

This audit checks the main data quality issues before modelling:

| Check | Why it matters |
|---|---|
| Missing values | Rows with missing target values cannot be used for supervised learning. Missing input values must be imputed, removed, or handled before model fitting. |
| Duplicate rows | Repeated rows can overweight the same sensor reading and bias validation results. |
| Status spelling variants | The `Status` feature is categorical, so spelling variants such as `subcool` or `saperheated` need to map to consistent numeric values. |
| IQR outlier counts | The IQR check gives a quick view of unusually large/small values. I do not blindly delete all outliers, because high pressure values can be physically meaningful in BLEVE data. |
| Target distribution | The target pressure is strongly right-skewed: most pressure values are small, but a few cases are much larger. This is important because large-pressure cases can have a strong influence on model fitting and error analysis. |

In [3]:
# Load raw files first so the audit reflects the original data.
train_audit = pd.read_csv(TRAIN_PATH)
test_audit = pd.read_csv(TEST_PATH)

# Basic shape, duplicate, and missing-value checks.
print("Train shape:", train_audit.shape)
print("Test shape:", test_audit.shape)
print("Duplicate train rows:", train_audit.duplicated().sum())
print("Duplicate test rows:", test_audit.duplicated().sum())
print("Total missing train values:", int(train_audit.isna().sum().sum()))
print("Total missing test values:", int(test_audit.isna().sum().sum()))
print("Rows with missing values in train:", int(train_audit.isna().any(axis=1).sum()))
print("Rows with missing values in test:", int(test_audit.isna().any(axis=1).sum()))

# Check spelling variants in the categorical Status column.
print("\nStatus value counts in train:")
print(train_audit["Status"].astype(str).str.lower().str.replace(" ", "", regex=False).value_counts())

# Use IQR counts as a quick outlier diagnostic for numeric inputs.
num_cols = train_audit.select_dtypes(include=np.number).columns.drop(TARGET_COL, errors="ignore")
outlier_counts = {}
for col in num_cols:
    q1, q3 = train_audit[col].quantile([0.25, 0.75])
    iqr = q3 - q1
    outlier_counts[col] = int(((train_audit[col] < q1 - 1.5 * iqr) | (train_audit[col] > q3 + 1.5 * iqr)).sum())

print("\nTop IQR outlier counts:")
print(pd.Series(outlier_counts).sort_values(ascending=False).head(10))

print("\nTarget pressure summary:")
print(train_audit[TARGET_COL].describe())

Train shape: (10050, 25)
Test shape: (3203, 24)
Duplicate train rows: 50
Duplicate test rows: 0
Total missing train values: 302
Total missing test values: 0
Rows with missing values in train: 112
Rows with missing values in test: 0

Status value counts in train:
Status
subcooled      6305
superheated    3689
subcool          23
subcoled         14
superheat         8
saperheated       4
Name: count, dtype: int64

Top IQR outlier counts:
Sensor Position z                  92
Vapour Height (m)                  71
Tank Failure Pressure (bar)        50
Sensor Position y                  10
Obstacle Thickness (m)              0
Sensor Position x                   0
Sensor Position Side                0
Sensor ID                           0
Liquid Critical Temperature (C)     0
Liquid Boiling Temperature (C)      0
dtype: int64

Target pressure summary:
count    10040.000000
mean         0.360492
std          0.495271
min          0.016098
25%          0.102211
50%          0.206346
75%     

## 3. Event Assignment and Denoising

The training file is row-based, but the physical experiment is event-based. One BLEVE event is one explosion scenario, and that scenario is measured by up to 27 sensors. So the rows are related in groups of 27.

Example structure:

```text
Event 0: rows 0  to 26   -> 27 sensor readings from the same BLEVE setup
Event 1: rows 27 to 53   -> 27 sensor readings from the next BLEVE setup
Event 2: rows 54 to 80   -> 27 sensor readings from another BLEVE setup
```

The code creates two helper columns from `ID`:

| Column | Code idea | Meaning |
|---|---|---|
| `Event_ID` | `ID // 27` | Which BLEVE event this row belongs to. `//` gives the full group number. |
| `Event_Pos` | `ID % 27` | The zero-based offset/position inside that event. `%` gives the remainder after grouping. |

Important distinction: `Event_Pos` is an internal offset from `0` to `26`, but the actual `Sensor ID` values in the dataset are from `1` to `27`. The offset is used only to locate a row inside a 27-row event group; it does not rename the sensors.

For example:

| `ID` | `ID // 27` | `ID % 27` | Meaning |
|---:|---:|---:|---|
| 0 | 0 | 0 | Event 0, offset 0 |
| 26 | 0 | 26 | Event 0, offset 26 |
| 27 | 1 | 0 | Event 1, offset 0 |
| 53 | 1 | 26 | Event 1, offset 26 |
| 54 | 2 | 0 | Event 2, offset 0 |

This separation is useful for three reasons:

1. **Cleaning shared setup values**  
   Rows from the same event should share the same tank dimensions, liquid ratio, temperatures, obstacle dimensions, and obstacle distance. If one sensor row has a slightly different setup value, it is likely noise or a data entry issue.

2. **Keeping sensor-specific values separate**  
   Sensor position, sensor side, sensor ID, and target pressure are expected to differ between sensors, so these are not overwritten by event medians.

3. **Fairer validation later**  
   Event grouping lets the notebook validate on complete unseen BLEVE scenarios instead of randomly mixing sensors from the same event into both training and validation.

Denoising means replacing event-level setup variables with the median value inside the same `Event_ID`.

Example:

```text
Event 5 tank width values before denoising:
2.0, 2.0, 2.0, 2.1, 2.0

Median value:
2.0

Event 5 tank width values after denoising:
2.0, 2.0, 2.0, 2.0, 2.0
```

The median is used instead of the mean because it is more robust to unusual values. A single strange value has less effect on a median.

In short: event assignment tells the notebook which rows belong to the same BLEVE scenario, and denoising makes the shared physical setup values consistent within each scenario.

An additional issue in this dataset is that the train/test boundary cuts through an event. The test file ends with the first part of one event, and the training file starts with the remaining sensors from that same event. To avoid training on only the tail end of a test event, the training preparation later keeps only complete 27-sensor training events. The test set is not filtered, because Kaggle requires a prediction for every test row.


In [4]:
# Create event groups and smooth event-level setup variables.
def apply_events_and_denoise(df):
    df = df.copy()

    # Convert ID and Sensor ID into numeric helper columns.
    df["ID_Num"] = pd.to_numeric(df["ID"], errors="coerce") if "ID" in df.columns else np.arange(len(df), dtype=float)
    if "Sensor ID" in df.columns:
        df["Sensor_ID_Num"] = pd.to_numeric(
            df["Sensor ID"].astype(str).str.extract(r"(-?\d+\.?\d*)")[0],
            errors="coerce",
        )
    else:
        df["Sensor_ID_Num"] = np.nan

    # ID // 27 gives the event; ID % 27 gives the zero-based offset within the event.
    valid = df["ID_Num"].notna()
    df["Event_ID"], df["Event_Pos"] = np.nan, np.nan

    if valid.any():
        df.loc[valid, "Event_ID"] = (df.loc[valid, "ID_Num"].astype(int) // EVENT_SIZE).astype(int)
        df.loc[valid, "Event_Pos"] = (df.loc[valid, "ID_Num"].astype(int) % EVENT_SIZE).astype(int)

    # Fallback for any rows where ID is unavailable.
    if (~valid).any():
        reset = df["Sensor_ID_Num"].diff().fillna(0) < 0
        offset = int(np.nanmax(df["Event_ID"].values)) if valid.any() else -1
        df.loc[~valid, "Event_ID"] = reset.cumsum().astype(int).loc[~valid] + offset + 1
        df.loc[~valid, "Event_Pos"] = df.groupby(reset.cumsum().astype(int)).cumcount().loc[~valid]

    df["Event_ID"] = df["Event_ID"].astype(int)

    # Only smooth shared setup variables; keep sensor-specific and target columns unchanged.
    skip = {
        "ID", "ID_Num", "Sensor ID", "Sensor_ID_Num", "Event_ID", "Event_Pos",
        "Sensor Position Side", "Sensor Position x", "Sensor Position y", "Sensor Position z", TARGET_COL,
    }
    for col in [c for c in df.select_dtypes(include=[np.number]).columns if c not in skip]:
        df[col] = df.groupby("Event_ID")[col].transform("median")

    # Status is also event-level, so use the modal value within each event.
    if "Status" in df.columns:
        df["Status"] = df.groupby("Event_ID")["Status"].transform(
            lambda x: x.mode()[0] if not x.mode().empty else (x.dropna().iloc[0] if len(x.dropna()) > 0 else np.nan)
        )

    return df


# Map noisy Status strings into a consistent binary numeric value.
def clean_status(x):
    match = get_close_matches(str(x).lower().replace(" ", ""), ["subcooled", "superheated"], n=1, cutoff=0.7)
    return 0.0 if match and match[0] == "subcooled" else (1.0 if match else np.nan)

## 4. Feature Engineering

The engineered features translate raw columns into quantities that are easier for the models to learn from.

| Feature group | Examples | Why it helps |
|---|---|---|
| Geometry | `Tank_Volume`, `Tank_Surface`, `Tank_Diagonal`, `Obstacle_Area` | Converts raw dimensions into size/area variables related to explosion energy and obstacle exposure. |
| Sensor distance | `Sensor_Distance`, `Scaled_Distance`, `Inv_Dist` | Pressure generally changes with distance from the source, so distance and inverse-distance terms are important. |
| Hopkinson-Cranz-inspired scaled distance | `Event_Strength_Proxy`, `Blast_Scaled_Distance`, `Inv_Blast_Scaled_Distance`, `Log_Blast_Scaled_Distance` | Uses a pressure-volume-liquid proxy with sensor distance. This follows the scaled-distance idea `Z = D / W^(1/3)`, but uses an event-strength proxy instead of explosive mass. |
| Thermodynamics | `Temp_Ratio`, `Reduced_Pressure`, `Superheat_Ratio`, `Vapour_Fraction` | Captures the liquid/vapour state and how close the substance is to critical or boiling conditions. |
| Obstacle interaction | `Obstacle_Blockage`, angle sine/cosine, angle-blockage interactions | Represents how the wall size, distance, and angle can affect reflection/deflection of the blast wave. |
| Sensor identity and side | `Side_1` to `Side_5`, `Sensor_1` to `Sensor_27`, sine/cosine encodings | Allows the model to learn systematic differences between sensor positions around the obstacle. |

The target pressure is positive and skewed, so the final model later uses log and Box-Cox style transformations. scikit-learn's `PowerTransformer` documentation describes these transformations as useful for reducing skewness and stabilising variance.
The scaled-distance feature is inspired by Hopkinson-Cranz blast scaling, commonly written as `Z = D / W^(1/3)`. In this notebook, `D` is sensor distance and `W` is replaced by `Event_Strength_Proxy`, because the dataset does not provide TNT-equivalent explosive mass. Therefore, this is a machine-learning proxy feature rather than an exact blast-physics calculation.


In [5]:
# Build physical, geometric, thermodynamic, sensor, and interaction features.
def engineer_features(df):
    d = df.copy()
    eps = 1e-6

    # Convert Status before creating interactions with thermodynamic features.
    if "Status" in d.columns:
        d["Status"] = d["Status"].apply(clean_status)

    # Correct pressure values that appear to be stored at a different scale.
    if "Tank Failure Pressure (bar)" in d.columns:
        tfp = pd.to_numeric(d["Tank Failure Pressure (bar)"], errors="coerce")
        med = tfp.median()
        if pd.notna(med) and med != 0:
            d["Tank Failure Pressure (bar)"] = np.where(tfp > med * 5, tfp / 100.0, tfp)

    # Force model inputs into numeric form where possible.
    for col in d.columns:
        if col not in ["ID", "Sensor ID", "Fluid_Key", TARGET_COL]:
            d[col] = pd.to_numeric(d[col], errors="coerce")

    # Geometry features describe tank size, obstacle size, and sensor distance.
    d["Tank_Volume (m3)"] = d["Tank Width (m)"] * d["Tank Length (m)"] * d["Tank Height (m)"]
    d["Tank_Surface"] = 2 * (
        d["Tank Width (m)"] * d["Tank Length (m)"]
        + d["Tank Width (m)"] * d["Tank Height (m)"]
        + d["Tank Length (m)"] * d["Tank Height (m)"]
    )
    d["Obstacle_Area"] = d["Obstacle Width (m)"] * d["Obstacle Height (m)"]
    d["Obstacle_Volume_Proxy"] = d["Obstacle_Area"] * d.get("Obstacle Thickness (m)", 1.0)
    d["Obstacle_Thinness"] = d.get("Obstacle Thickness (m)", 0.0) / (d["Obstacle Width (m)"] + eps)
    d["Sensor_Distance"] = np.sqrt(d["Sensor Position x"]**2 + d["Sensor Position y"]**2 + d["Sensor Position z"]**2)
    d["Tank_Diagonal"] = np.sqrt(d["Tank Width (m)"]**2 + d["Tank Length (m)"]**2 + d["Tank Height (m)"]**2)

    d["Sensor_Distance_Norm_Diagonal"] = d["Sensor_Distance"] / (d["Tank_Diagonal"] + eps)
    d["Sensor_X_Norm_Length"] = d["Sensor Position x"] / (d["Tank Length (m)"] + eps)
    d["Sensor_Y_Norm_Width"] = d["Sensor Position y"] / (d["Tank Width (m)"] + eps)
    d["Sensor_Z_Norm_Height"] = d["Sensor Position z"] / (d["Tank Height (m)"] + eps)

    d["Abs_X"] = d["Sensor Position x"].abs()
    d["Abs_Y"] = d["Sensor Position y"].abs()
    d["Abs_Z"] = d["Sensor Position z"].abs()
    d["X_Sq"] = d["Sensor Position x"]**2
    d["Y_Sq"] = d["Sensor Position y"]**2
    d["Z_Sq"] = d["Sensor Position z"]**2
    d["XY_Product"] = d["Sensor Position x"] * d["Sensor Position y"]
    d["XZ_Product"] = d["Sensor Position x"] * d["Sensor Position z"]
    d["YZ_Product"] = d["Sensor Position y"] * d["Sensor Position z"]

    d["Aspect_Ratio_LW_Bounded"] = np.tanh(d["Tank Length (m)"] / (d["Tank Width (m)"] + eps))
    d["Aspect_Ratio_HW_Bounded"] = np.tanh(d["Tank Height (m)"] / (d["Tank Width (m)"] + eps))
    d["Vol_Surf_Ratio_Bounded"] = np.tanh(d["Tank_Volume (m3)"] / (d["Tank_Surface"] + eps))
    d["Obstacle_Dist_Ratio_Bounded"] = np.tanh(d["Sensor_Distance"] / (d["Obstacle Distance to BLEVE (m)"] + eps))

    # Hopkinson-Cranz-inspired scaled-distance features combine an event-strength proxy and sensor distance.
    d["Energy_Proxy"] = d["Tank Failure Pressure (bar)"] * d["Tank_Volume (m3)"]
    d["Pressure_Liquid_Proxy"] = d["Tank Failure Pressure (bar)"] * d["Liquid Ratio (%)"]
    d["Event_Strength_Proxy"] = d["Energy_Proxy"] * (d["Liquid Ratio (%)"] / 100.0 + 1e-3)
    d["Blast_Scaled_Distance"] = d["Sensor_Distance"] / (np.cbrt(np.clip(d["Event_Strength_Proxy"], 0, None)) + eps)
    d["Log_Blast_Scaled_Distance"] = np.log1p(d["Blast_Scaled_Distance"])
    d["Inv_Blast_Scaled_Distance"] = 1.0 / (d["Blast_Scaled_Distance"] + eps)
    d["Inv_Blast_Scaled_Distance_Sq"] = 1.0 / (d["Blast_Scaled_Distance"] ** 2 + eps)
    d["Inv_Blast_Scaled_Distance_Cube"] = 1.0 / (d["Blast_Scaled_Distance"] ** 3 + eps)

    d["Angle_XY"] = np.arctan2(d["Sensor Position y"], d["Sensor Position x"] + eps)
    d["Angle_Z"] = np.arccos(np.clip(d["Sensor Position z"] / (d["Sensor_Distance"] + eps), -1.0, 1.0))
    d["Scaled_Distance"] = d["Sensor_Distance"] / (np.cbrt(d["Tank_Volume (m3)"]) + eps)
    d["Inv_Dist"] = 1.0 / (d["Sensor_Distance"] + eps)

    # Thermodynamic ratios capture liquid/vapour state and critical conditions.
    ct_k = d["Liquid Critical Temperature (C)"] + 273.15
    bt_k = d["Liquid Boiling Temperature (C)"] + 273.15
    d["Temp_Diff"] = d["Vapour Temperature (K)"] - d["Liquid Temperature (K)"]
    d["Temp_Ratio"] = d["Vapour Temperature (K)"] / (d["Liquid Temperature (K)"] + eps)
    d["Vapour_Fraction"] = d["Vapour Height (m)"] / (d["Tank Height (m)"] + eps)
    d["Liquid_Ratio_Frac"] = d["Liquid Ratio (%)"] / 100.0
    d["Reduced_Pressure"] = d["Tank Failure Pressure (bar)"] / (d["Liquid Critical Pressure (bar)"] + eps)
    d["Reduced_Liquid_Temp"] = d["Liquid Temperature (K)"] / (ct_k + eps)
    d["Reduced_Vapour_Temp"] = d["Vapour Temperature (K)"] / (ct_k + eps)
    d["Superheat_Ratio"] = d["Liquid Temperature (K)"] / (bt_k + eps)

    # Obstacle features capture wall blockage and orientation effects.
    d["Obstacle_Blockage"] = d["Obstacle_Area"] / (d["Obstacle Distance to BLEVE (m)"] + eps)
    d["Obstacle_Blockage_Volume"] = d["Obstacle_Volume_Proxy"] / (d["Obstacle Distance to BLEVE (m)"] + eps)
    rads = np.deg2rad(d["Obstacle Angle"].fillna(0))
    d["Obstacle_Angle_Sin"] = np.sin(rads)
    d["Obstacle_Angle_Cos"] = np.cos(rads)

    # Interaction terms expose nonlinear combinations to the models.
    d["Status_x_Superheat"] = d["Status"] * d["Superheat_Ratio"]
    d["Status_x_Reduced_Pressure"] = d["Status"] * d["Reduced_Pressure"]
    d["Obstacle_Angle_x_Blockage"] = d["Obstacle_Angle_Sin"] * d["Obstacle_Blockage"]
    d["Inv_Blast_Scaled_Distance_x_Temp_Ratio"] = d["Inv_Blast_Scaled_Distance"] * d["Temp_Ratio"]
    d["Inv_Blast_Scaled_Distance_x_Obstacle_Blockage"] = d["Inv_Blast_Scaled_Distance"] * d["Obstacle_Blockage"]
    d["Inv_Blast_Scaled_Distance_x_Vapour_Fraction"] = d["Inv_Blast_Scaled_Distance"] * d["Vapour_Fraction"]
    d["Log_Blast_Scaled_Distance_x_Superheat"] = d["Log_Blast_Scaled_Distance"] * d["Superheat_Ratio"]
    d["Log_Blast_Scaled_Distance_x_Reduced_Pressure"] = d["Log_Blast_Scaled_Distance"] * d["Reduced_Pressure"]
    d["Obstacle_Angle_Cos_x_Inv_Blast_Scaled_Distance"] = d["Obstacle_Angle_Cos"] * d["Inv_Blast_Scaled_Distance"]
    d["Obstacle_Angle_Sin_x_Inv_Blast_Scaled_Distance"] = d["Obstacle_Angle_Sin"] * d["Inv_Blast_Scaled_Distance"]
    d["Status_x_Inv_Blast_Scaled_Distance"] = d["Status"] * d["Inv_Blast_Scaled_Distance"]
    d["Status_x_Obstacle_Blockage"] = d["Status"] * d["Obstacle_Blockage"]
    d["Sensor_Z_x_Inv_Blast_Scaled_Distance"] = d["Sensor_Z_Norm_Height"] * d["Inv_Blast_Scaled_Distance"]
    d["Sensor_Y_x_Obstacle_Blockage"] = d["Sensor_Y_Norm_Width"] * d["Obstacle_Blockage"]

    # One-hot side and sensor features let the model learn repeated position effects.
    side_num = pd.to_numeric(d["Sensor Position Side"], errors="coerce")
    for s in [1, 2, 3, 4, 5]:
        d[f"Side_{s}"] = (side_num == s).astype(float)
        d[f"Side_{s}_Inv_Blast_Scaled_Distance"] = d[f"Side_{s}"] * d["Inv_Blast_Scaled_Distance"]
        d[f"Side_{s}_Log_Blast_Scaled_Distance"] = d[f"Side_{s}"] * d["Log_Blast_Scaled_Distance"]
        d[f"Side_{s}_Obstacle_Blockage"] = d[f"Side_{s}"] * d["Obstacle_Blockage"]
        d[f"Side_{s}_Scaled_Distance"] = d[f"Side_{s}"] * d["Scaled_Distance"]

    sid = pd.to_numeric(d["Sensor_ID_Num"], errors="coerce")
    for s in range(1, 28):
        d[f"Sensor_{s}"] = (sid == s).astype(float)
    d["Sensor_ID_Sin"] = np.sin(2 * np.pi * sid / 27.0)
    d["Sensor_ID_Cos"] = np.cos(2 * np.pi * sid / 27.0)

    pos = pd.to_numeric(d["Event_Pos"], errors="coerce")
    d["Event_Pos_Sin"] = np.sin(2 * np.pi * pos / 27.0)
    d["Event_Pos_Cos"] = np.cos(2 * np.pi * pos / 27.0)

    # Log versions reduce skew for strictly positive engineered quantities.
    log_cols = [
        "Tank_Volume (m3)", "Tank_Surface", "Obstacle_Area", "Obstacle_Volume_Proxy",
        "Sensor_Distance", "Tank_Diagonal", "Energy_Proxy", "Event_Strength_Proxy",
        "Inv_Blast_Scaled_Distance", "Inv_Blast_Scaled_Distance_Sq", "Inv_Blast_Scaled_Distance_Cube",
    ]
    for c in log_cols:
        d[f"{c}_log"] = np.log1p(np.clip(d[c], 0, None))

    # Fluid classes are inferred from critical pressure and temperature.
    d["Fluid_Key"] = (
        d["Liquid Critical Pressure (bar)"].round(3).astype(str)
        + "_"
        + d["Liquid Critical Temperature (C)"].round(3).astype(str)
    )
    return d.replace([np.inf, -np.inf], np.nan)

## 5. Prepare Train/Test Matrices

Rows without an ID or training target are removed, exact duplicate rows are removed, important geometry/sensor fields are required, event features are created, and a shared numeric feature list is built for both train and test.

A complete training event should contain all 27 sensor positions. After duplicate and missing-target removal, a few training events are incomplete. The most important incomplete event is the boundary event split across test and train. For training, I keep only complete events where:

```text
number of rows in the event = 27
number of unique Event_Pos offsets = 27
minimum Event_Pos offset = 0
maximum Event_Pos offset = 26
```

This avoids learning from partial events that do not represent a full BLEVE sensor layout. The actual `Sensor ID` column is still kept as sensors `1` to `27`. The test set is kept intact because every test row needs a prediction.

In [6]:
# Keep only full 27-row training events after cleaning.
def keep_complete_training_events(df):
    df = df.copy()
    # Summarise each event to detect partial event groups.
    event_summary = df.groupby("Event_ID").agg(
        row_count=("Event_Pos", "size"),
        unique_positions=("Event_Pos", "nunique"),
        min_position=("Event_Pos", "min"),
        max_position=("Event_Pos", "max"),
    )
    # A complete event must contain all offsets from 0 to 26 exactly once.
    complete_event_ids = event_summary[
        (event_summary["row_count"] == EVENT_SIZE)
        & (event_summary["unique_positions"] == EVENT_SIZE)
        & (event_summary["min_position"] == 0)
        & (event_summary["max_position"] == EVENT_SIZE - 1)
    ].index

    removed_events = len(event_summary) - len(complete_event_ids)
    removed_rows = len(df) - int(df["Event_ID"].isin(complete_event_ids).sum())
    print(f"Incomplete training events removed: {removed_events}")
    print(f"Rows removed from incomplete training events: {removed_rows}")

    return df[df["Event_ID"].isin(complete_event_ids)].reset_index(drop=True)


# Load, de-duplicate, sort, and denoise before feature engineering.
print("1. Loading and denoising data...")
train_input = (
    pd.read_csv(TRAIN_PATH)
    .drop_duplicates()
    .dropna(subset=["ID", TARGET_COL])
    .sort_values("ID")
    .reset_index(drop=True)
)
test_input = (
    pd.read_csv(TEST_PATH)
    .drop_duplicates()
    .dropna(subset=["ID"])
    .sort_values("ID")
    .reset_index(drop=True)
)

# Create event features for both datasets; only training is filtered for completeness.
tr_df = apply_events_and_denoise(train_input)
te_df = apply_events_and_denoise(test_input)

# Require the core geometry/sensor columns needed by engineered features.
critical = ["Tank Width (m)", "Tank Length (m)", "Tank Height (m)", "Sensor Position x", "Sensor Position y", "Sensor Position z", TARGET_COL]
tr_df = tr_df.dropna(subset=[c for c in critical if c in tr_df.columns]).dropna(thresh=int(0.5 * tr_df.shape[1])).reset_index(drop=True)
tr_df = keep_complete_training_events(tr_df)

# Engineer the same feature space for train and test.
tr_raw = engineer_features(tr_df)
te_raw = engineer_features(te_df)

# Add matching one-hot fluid-class columns to both datasets.
for key in sorted(pd.concat([tr_raw["Fluid_Key"], te_raw["Fluid_Key"]]).dropna().unique()):
    tr_raw[f"Fluid_Class_{key}"] = (tr_raw["Fluid_Key"] == key).astype(float)
    te_raw[f"Fluid_Class_{key}"] = (te_raw["Fluid_Key"] == key).astype(float)

# Main numeric feature list used by neural network and tree branches.
feat = [
    c for c in tr_raw.select_dtypes(include=[np.number]).columns
    if c not in {TARGET_COL, "ID", "ID_Num", "Sensor ID", "Event_ID", "Fluid_Key", "Sensor Position Side"}
]
# Smaller physics-focused feature list used by the residual Ridge branch.
phys = [c for c in PHYSICS_COLS if c in tr_raw.columns] + [c for c in tr_raw.columns if c.startswith("Fluid_Class_")]

print("Rows after cleaning:", len(tr_raw))
print("Engineered numeric feature count:", len(feat))
print("Physics/residual branch feature count:", len(phys))
print("Train event count:", tr_raw["Event_ID"].nunique())
print("Test event count:", te_raw["Event_ID"].nunique())
print("Remaining missing target values:", int(tr_raw[TARGET_COL].isna().sum()))

1. Loading and denoising data...
Incomplete training events removed: 11
Rows removed from incomplete training events: 270
Rows after cleaning: 9720
Engineered numeric feature count: 154
Physics/residual branch feature count: 35
Train event count: 360
Test event count: 119
Remaining missing target values: 0


## 6. Model Definitions

The notebook compares several model families because each one learns a different type of relationship.

| Model | Role | Strength | Limitation |
|---|---|---|---|
| Ridge regression | Linear baseline | Simple, stable, interpretable, and regularised. Useful for broad linear trends. | Cannot capture complex nonlinear pressure interactions by itself. |
| Random Forest | Tree-based baseline | Captures nonlinear rules and feature thresholds without scaling. Averaging many trees helps reduce overfitting. | Can struggle to extrapolate smoothly and may not capture all continuous physical trends. |
| Neural Network | Nonlinear baseline | Learns smooth nonlinear relationships between engineered features and pressure. | Sensitive to scaling and hyperparameters; needs early stopping/regularisation. |
| Final ensemble | Main model | Combines transformed-target MLPs, residual learning, log-target MLPs, and histogram gradient boosting. | More computationally expensive and less directly interpretable. |

The final model has several branches:

| Branch | What it does |
|---|---|
| Box-Cox MLP branch | Transforms the target to reduce skew, trains several MLPs, then transforms predictions back to pressure units. |
| Residual branch | Fits a small physics-focused Ridge model first, then trains MLPs on the remaining error. This lets the neural network focus on nonlinear corrections instead of relearning the whole trend. |
| Log-target MLP branch | Models `log(pressure)`, which helps when relative error matters and the target is positive/skewed. |
| Histogram Gradient Boosting branch | Adds a tree-boosting view of the problem, useful for nonlinear thresholds and interactions. |
| Seed averaging | Trains multiple versions with different random seeds and averages them to reduce sensitivity to one lucky/unlucky initialisation. |

Residual learning in plain words:

```text
actual pressure = simple trend + leftover error
leftover error  = actual pressure - Ridge prediction
final branch    = Ridge prediction + neural network prediction of leftover error
```

This is useful because the Ridge model can capture broad physical trends, while the neural networks can focus on more complicated sensor, obstacle, and thermodynamic effects. scikit-learn documentation is used for `RidgeCV`, `MLPRegressor`, `RandomForestRegressor`, and `HistGradientBoostingRegressor`.

In [7]:
# Return the MLP architectures used by each ensemble branch.
def get_mlps(seed, branch):
    params = dict(
        learning_rate="adaptive",
        max_iter=2200,
        early_stopping=True,
        validation_fraction=0.08,
        n_iter_no_change=60,
    )
    # Box-Cox branch uses three MLP shapes and a weighted average.
    if branch == "boxcox":
        return [
            MLPRegressor(hidden_layer_sizes=(256, 128, 128, 64, 32), activation="relu", alpha=0.01, random_state=seed, **params),
            MLPRegressor(hidden_layer_sizes=(512, 256, 64), activation="relu", alpha=0.005, random_state=seed + 1, **params),
            MLPRegressor(hidden_layer_sizes=(256, 128, 64), activation="tanh", alpha=0.002, random_state=seed + 2, **params),
        ]

    params["max_iter"] = 1800
    params["n_iter_no_change"] = 50
    return [
        MLPRegressor(hidden_layer_sizes=(384, 192, 96, 48), activation="relu", alpha=0.006, random_state=seed, **params),
        MLPRegressor(hidden_layer_sizes=(256, 128, 64), activation="tanh", alpha=0.003, random_state=seed + 11, **params),
    ]


# Train several MLPs across seeds, then average their predictions.
def train_mlp_batch(X_tr, y_tr, X_te, seeds, weights, branch):
    # Standardise the transformed target so MLP optimisation is more stable.
    y_mean = y_tr.mean()
    y_std = max(y_tr.std(), 1e-8)
    y_scaled = (y_tr - y_mean) / y_std

    # Seed averaging reduces dependence on one random initialisation.
    all_preds = []
    for seed in seeds:
        model_preds = [model.fit(X_tr, y_scaled).predict(X_te) for model in get_mlps(seed, branch)]
        if weights is None:
            all_preds.append(np.mean(model_preds, axis=0))
        else:
            all_preds.append(np.average(model_preds, axis=0, weights=weights))

    return np.mean(all_preds, axis=0) * y_std + y_mean


# Train the histogram gradient boosting branch on log pressure.
def train_hgb(X_tr, y_tr, X_te, seeds):
    y_log = np.log(np.clip(y_tr, 1e-9, None))
    y_mean = y_log.mean()
    y_std = max(y_log.std(), 1e-8)

    preds = []
    for seed in seeds:
        model = HistGradientBoostingRegressor(
            learning_rate=0.035,
            max_iter=700,
            max_leaf_nodes=31,
            min_samples_leaf=20,
            l2_regularization=0.03,
            early_stopping=True,
            random_state=seed,
        )
        model.fit(X_tr, (y_log - y_mean) / y_std)
        preds.append(model.predict(X_te))

    return np.exp(np.mean(preds, axis=0) * y_std + y_mean)

## 7. Calibration Functions

Calibration is a final correction step learned from the validation split. The model first makes normal predictions. Then the calibration code checks whether those predictions have systematic bias in different groups.

The main idea is:

```text
calibration ratio = actual pressure / predicted pressure
```

If a group has a median ratio above `1.0`, the model is under-predicting for that group. If the median ratio is below `1.0`, the model is over-predicting for that group.

The calibration function learns small multiplicative corrections for:

| Calibration level | What it checks | Example interpretation |
|---|---|---|
| Global factor | Overall bias across the validation set | The whole model is slightly too high or too low. |
| Sensor side factor | Bias for each obstacle side | Sensors on one side of the wall may be systematically harder to predict. |
| Sensor ID factor | Bias for each sensor index | A specific sensor position may have a repeated pattern. |
| Prediction-bin factor | Bias for low/medium/high predicted pressures | Very small or very large pressures may need slightly different correction. |

The factors are clipped to narrow ranges such as `0.975` to `1.025`. This is intentional: calibration should make small bias corrections, not replace the model. The validation search tries several correction strengths and keeps the combination with the lowest validation MAPE.

In [8]:
# Apply learned multiplicative calibration factors to predictions.
def apply_calibration(pred, raw_df, cal):
    p = pred * cal["g"]
    sides = pd.to_numeric(raw_df["Sensor Position Side"], errors="coerce").values
    sensors = pd.to_numeric(raw_df["Sensor_ID_Num"], errors="coerce").values

    # Correct systematic bias by side and sensor ID.
    for side, factor in cal["s"].items():
        p[sides == side] *= factor
    for sensor, factor in cal["sen"].items():
        p[sensors == sensor] *= factor

    # Correct bias by prediction magnitude bin.
    if cal["b"]:
        for i, factor in enumerate(cal["b"]):
            p[(p >= cal["e"][i]) & (p <= cal["e"][i + 1])] *= factor

    return np.clip(p, 1e-9, None)


# Search for small calibration factors that minimise validation MAPE.
def learn_calibration(y_true, pred, raw_df):
    best_cal, best_score = None, float("inf")
    # Ratio above 1 means under-prediction; below 1 means over-prediction.
    ratio = np.clip(y_true / np.clip(pred, 1e-9, None), 0.02, 0.98)
    sides = pd.to_numeric(raw_df["Sensor Position Side"], errors="coerce").values
    sensors = pd.to_numeric(raw_df["Sensor_ID_Num"], errors="coerce").values

    # Try conservative correction strengths and keep the best validation score.
    for global_strength in [0.0, 0.1, 0.2, 0.3, 0.4]:
        for side_strength in [0.0, 0.1, 0.2, 0.3]:
            for sensor_strength in [0.0, 0.1, 0.2, 0.3, 0.4]:
                for bin_strength in [0.0, 0.1, 0.2]:
                    for n_bins in [6, 8]:
                        cal = {
                            "g": float(np.clip(1.0 + global_strength * (np.median(ratio) - 1.0), 0.985, 1.015)),
                            "s": {},
                            "sen": {},
                            "b": [],
                            "e": np.unique(np.quantile(pred, np.linspace(0.0, 1.0, n_bins + 1))),
                        }

                        for side in [1, 2, 3, 4, 5]:
                            mask = sides == side
                            cal["s"][float(side)] = float(np.clip(1.0 + side_strength * (np.median(ratio[mask]) - 1.0), 0.975, 1.025)) if mask.sum() >= 50 else 1.0

                        for sensor in range(1, 28):
                            mask = sensors == sensor
                            cal["sen"][float(sensor)] = float(np.clip(1.0 + sensor_strength * (np.median(ratio[mask]) - 1.0), 0.980, 1.020)) if mask.sum() >= 40 else 1.0

                        if len(cal["e"]) > 2:
                            for i in range(len(cal["e"]) - 1):
                                mask = (pred >= cal["e"][i]) & (pred <= cal["e"][i + 1])
                                cal["b"].append(float(np.clip(1.0 + bin_strength * (np.median(ratio[mask]) - 1.0), 0.985, 1.015)) if mask.sum() >= 80 else 1.0)

                        score = mean_absolute_percentage_error(y_true, apply_calibration(pred.copy(), raw_df, cal))
                        if score < best_score:
                            best_score, best_cal = score, cal

    return best_cal if best_cal else {"g": 1.0, "s": {}, "sen": {}, "b": [], "e": []}

## 8. Final Ensemble Runner

The `run_models` function is the main training/prediction engine. It performs the following sequence:

1. **Median imputation:** missing numeric feature values are filled using training medians.
2. **Robust scaling:** the main feature matrix is scaled with `RobustScaler`, which is less sensitive to outliers than standard scaling because it uses median and interquartile range information.
3. **Physics-feature scaling:** the smaller physics feature set is standardised for the Ridge residual branch.
4. **Box-Cox target transformation:** this is where the Box-Cox transform is actually applied. Pressure is transformed into a more model-friendly distribution before the Box-Cox MLP branch is trained.
5. **Ridge residual modelling:** Ridge learns the broad physics trend, then MLPs learn the remaining residuals.
6. **Side-aware blending:** the plain Box-Cox branch and residual branch are blended with side-specific weights.
7. **Log-target and HGB branches:** this is where the log transform is actually applied for the log-target MLP branch. Extra predictions are also generated from histogram gradient boosting.
8. **Validation weight search:** during validation, the code searches a blend weight between the Box-Cox and log branches and keeps the best validation MAPE.

This structure is more advanced than a single model, but each branch has a specific job: stabilise the target, learn broad trends, learn nonlinear residuals, add tree-based behaviour, and reduce randomness through seed averaging.

In [9]:
# Train the final ensemble branches and return predictions.
def run_models(X_tr_raw, y_tr, X_te_raw, is_val=False):
    # Fill missing feature values using training medians only.
    med = X_tr_raw[feat].median().fillna(0.0)

    # Robust scaling protects MLPs from large outlier influence.
    scaler_all = RobustScaler()
    X_tr_scaled = scaler_all.fit_transform(X_tr_raw[feat].fillna(med))
    X_te_scaled = scaler_all.transform(X_te_raw[feat].fillna(med))

    # Standard scaling is used for the compact physics branch.
    scaler_phys = StandardScaler()
    X_tr_phys = scaler_phys.fit_transform(X_tr_raw[phys].fillna(0.0))
    X_te_phys = scaler_phys.transform(X_te_raw[phys].fillna(0.0))

    # Transform the positive target to reduce skew for the Box-Cox branch.
    boxcox = PowerTransformer(method="box-cox")
    y_offset = max(0.0, 1e-6 - np.nanmin(y_tr))
    y_box = boxcox.fit_transform((y_tr + y_offset).values.reshape(-1, 1)).ravel()

    # Ridge learns the broad physics trend; MLPs learn the residual error.
    ridge = RidgeCV(alphas=[0.1, 0.5, 1.0, 2.0, 5.0, 10.0]).fit(X_tr_phys, y_box)
    residual = y_box - ridge.predict(X_tr_phys)

    # Plain and residual Box-Cox predictions are blended by sensor side.
    plain = train_mlp_batch(X_tr_scaled, y_box, X_te_scaled, MAIN_SEEDS, PLAIN_W, "boxcox")
    resid = ridge.predict(X_te_phys) + train_mlp_batch(X_tr_scaled, residual, X_te_scaled, MAIN_SEEDS, RESID_W, "boxcox")

    sides = pd.to_numeric(X_te_raw["Sensor Position Side"], errors="coerce").values
    blend = np.zeros_like(plain)
    for i, side in enumerate(sides):
        alpha = SIDE_ALPHA.get(side, 0.76)
        blend[i] = (1.0 - alpha) * plain[i] + alpha * resid[i]

    # Convert predictions back to pressure units and add the log/HGB branches.
    box_pred = np.clip(boxcox.inverse_transform(blend.reshape(-1, 1)).ravel() - y_offset, 1e-9, None)
    log_pred = train_mlp_batch(X_tr_scaled, np.log(np.clip(y_tr, 1e-9, None)), X_te_scaled, LOG_SEEDS, None, "log")
    hgb_pred = train_hgb(X_tr_scaled, y_tr, X_te_scaled, HGB_SEEDS)

    # During validation, tune the blend between Box-Cox and log-target branches.
    if is_val:
        best_weight, best_score = 1.0, float("inf")
        for weight in np.linspace(0.65, 1.00, 15):
            score = mean_absolute_percentage_error(X_te_raw[TARGET_COL], weight * box_pred + (1.0 - weight) * np.exp(log_pred))
            if score < best_score:
                best_score, best_weight = score, weight
        return best_weight, box_pred, np.exp(log_pred), hgb_pred

    return box_pred, np.exp(log_pred), hgb_pred

## 9. Event-Based Validation and Model Comparison

This notebook uses event-based hold-out validation. A single validation set is created by holding out complete BLEVE events rather than randomly splitting individual sensor rows. This is important because rows are grouped by BLEVE event, and sensors from the same event share the same physical setup.

A random row split can accidentally put this structure into both training and validation:

```text
Training set:   Event 10, sensors 1-20
Validation set: Event 10, sensors 21-27
```

That would make validation too easy because the model has already seen most of the same event. Instead, this notebook keeps whole events together:

```text
Training set:   some complete events
Validation set: different complete events
```

The validation cell trains and compares:

| Model | What is measured |
|---|---|
| Linear Regression (Ridge) | Simple regularised linear baseline. |
| Random Forest | Tree-based nonlinear baseline. |
| Neural Network | Single tuned MLP baseline. |
| Final Ensemble Model | Full ensemble with target transforms, residual learning, boosting, and calibration. |

The metrics are:

| Metric | Meaning |
|---|---|
| MAPE | Mean absolute percentage error; lower is better. This is the most important metric for the Kaggle score. |
| R2 | Proportion of variance explained; closer to 1 is better. |
| RMSE | Penalises larger errors more strongly because errors are squared before averaging. |
| MAE | Average absolute error in pressure units. |

scikit-learn's regression metric documentation is used for the MAPE, R2, RMSE, and MAE calculations.

In [10]:
# Collect all validation metrics in a consistent dictionary.
def get_metrics(name, y_true, y_pred):
    return {
        "Model": name,
        "MAPE": mean_absolute_percentage_error(y_true, y_pred),
        "R2": r2_score(y_true, y_pred),
        "RMSE": np.sqrt(mean_squared_error(y_true, y_pred)),
        "MAE": mean_absolute_error(y_true, y_pred),
    }

print("2. Running validation phase...")
# Split by complete events, not random sensor rows.
events = np.sort(tr_raw["Event_ID"].unique())
val_mask = tr_raw["Event_ID"].isin(events[:min(te_raw["Event_ID"].nunique(), len(events) // 3)])

# Train the final model on training events and predict validation events.
best_w, v_box, v_log, v_hgb = run_models(tr_raw[~val_mask], tr_raw.loc[~val_mask, TARGET_COL], tr_raw[val_mask], is_val=True)
v_raw = (1.0 - HGB_W) * (best_w * v_box + (1.0 - best_w) * v_log) + HGB_W * v_hgb
# Learn calibration only from validation predictions.
cal = learn_calibration(tr_raw.loc[val_mask, TARGET_COL], v_raw, tr_raw[val_mask])
v_cal = apply_calibration(v_raw.copy(), tr_raw[val_mask], cal)
y_val_true = tr_raw.loc[val_mask, TARGET_COL]

print("Training simple baseline models for comparison...")
# Prepare the same event split for the three simpler comparison models.
med_simple = tr_raw[feat].median().fillna(0.0)
X_tr_simple = tr_raw[~val_mask][feat].fillna(med_simple)
X_val_simple = tr_raw[val_mask][feat].fillna(med_simple)
y_tr_simple = tr_raw.loc[~val_mask, TARGET_COL]

scaler_simple = StandardScaler()
X_tr_scaled_simple = scaler_simple.fit_transform(X_tr_simple)
X_val_scaled_simple = scaler_simple.transform(X_val_simple)

# Baseline 1: regularised linear model.
ridge_simple = Ridge(alpha=1.0, random_state=42).fit(X_tr_scaled_simple, y_tr_simple)
p_ridge = ridge_simple.predict(X_val_scaled_simple)

# Baseline 2: tree ensemble model.
rf_simple = RandomForestRegressor(
    n_estimators=1000,
    max_depth=30,
    min_samples_split=5,
    random_state=42,
    n_jobs=-1,
).fit(X_tr_simple, y_tr_simple)
p_rf = rf_simple.predict(X_val_simple)

# Baseline 3: single neural network model.
nn_simple = MLPRegressor(
    hidden_layer_sizes=(256, 256, 128, 128, 128, 64, 32),
    activation="relu",
    solver="adam",
    alpha=0.0039,
    learning_rate_init=0.00112,
    max_iter=2000,
    early_stopping=True,
    validation_fraction=0.1,
    n_iter_no_change=20,
    random_state=42,
).fit(X_tr_scaled_simple, y_tr_simple)
p_nn = nn_simple.predict(X_val_scaled_simple)

# Compare baselines against the final calibrated ensemble.
metrics = [
    get_metrics("Linear Regression (Ridge)", y_val_true, p_ridge),
    get_metrics("Random Forest", y_val_true, p_rf),
    get_metrics("Neural Network", y_val_true, p_nn),
    get_metrics("Final Ensemble Model", y_val_true, v_cal),
]

metrics_df = pd.DataFrame(metrics)
print("\n" + "=" * 80)
print(f"{'Model':<30} | {'MAPE':<10} | {'R2':<10} | {'RMSE':<10} | {'MAE':<10}")
print("-" * 80)
for row in metrics:
    print(f"{row['Model']:<30} | {row['MAPE']:<10.6f} | {row['R2']:<10.6f} | {row['RMSE']:<10.6f} | {row['MAE']:<10.6f}")
print("=" * 80 + "\n")

metrics_df

2. Running validation phase...
Training simple baseline models for comparison...

Model                          | MAPE       | R2         | RMSE       | MAE       
--------------------------------------------------------------------------------
Linear Regression (Ridge)      | 0.742062   | 0.781328   | 0.216587   | 0.132332  
Random Forest                  | 0.239264   | 0.853797   | 0.177098   | 0.076528  
Neural Network                 | 0.174315   | 0.950580   | 0.102964   | 0.048740  
Final Ensemble Model           | 0.126223   | 0.962758   | 0.089382   | 0.041311  



,Model,MAPE,R2,RMSE,MAE
0,Linear Regression (Ridge),0.742062,0.781328,0.216587,0.132332
1,Random Forest,0.239264,0.853797,0.177098,0.076528
2,Neural Network,0.174315,0.950580,0.102964,0.048740
3,Final Ensemble Model,0.126223,0.962758,0.089382,0.041311


## 10. Final Training and Prediction File

After validation, the final model is trained again using all labelled training rows. This gives the final model the maximum available labelled data before predicting the hidden test set.

The final prediction workflow is:

1. Train the final ensemble on all cleaned/engineered training data.
2. Predict the test rows using the same engineered feature set.
3. Apply the learned validation calibration factors.
4. Apply the final calibration strength in log space.
5. Save the Kaggle-format file as `prediction.csv` with columns `ID` and `Target Pressure (bar)`.

The test set does not contain target labels, so MAPE/R2/RMSE/MAE cannot be computed locally for test rows. The Kaggle score is the external test-set feedback.

In [11]:
# Retrain the final model using all labelled training events, then predict test.csv.
print("3. Training final model on the full dataset...")
t_box, t_log, t_hgb = run_models(tr_raw, tr_raw[TARGET_COL], te_raw)
# Reuse the best validation blend weight for the final test predictions.
t_raw = (1.0 - HGB_W) * (best_w * t_box + (1.0 - best_w) * t_log) + HGB_W * t_hgb

# Apply validation-learned calibration and save the Kaggle-format file.
t_cal = apply_calibration(t_raw.copy(), te_raw, cal)
log_raw = np.log(t_raw)
log_cal = np.log(t_cal)
t_final = np.clip(np.exp(log_raw + FINAL_CALIB_STRENGTH * (log_cal - log_raw)), 1e-9, None)

prediction = pd.DataFrame({"ID": te_df["ID"], TARGET_COL: t_final})
prediction.to_csv(OUTPUT_PATH, index=False)

print(f"Success. Output saved to {OUTPUT_PATH}")
print("Prediction shape:", prediction.shape)
print(prediction.head())
print("\nPrediction summary:")
print(prediction[TARGET_COL].describe())

3. Training final model on the full dataset...
Success. Output saved to prediction.csv
Prediction shape: (3203, 2)
   ID  Target Pressure (bar)
0   0               0.225285
1   1               0.238804
2   2               0.375165
3   3               0.312473
4   4               0.267041

Prediction summary:
count    3203.000000
mean        0.317446
std         0.433592
min         0.019168
25%         0.091555
50%         0.180839
75%         0.361801
max         6.758714
Name: Target Pressure (bar), dtype: float64


## 11. Reflection

The model improved as the workflow moved from raw features to physically meaningful engineered features and then from single models to a calibrated ensemble. Ridge gives a simple linear reference, Random Forest captures nonlinear tree patterns, and the neural network handles smoother nonlinear interactions. The final model combines target transformations, residual learning, tree boosting, seed averaging, side-aware blending, and calibration.

The main validation risk is leakage from sensors belonging to the same BLEVE event. Event-based validation reduces that risk by holding out complete scenarios together. This gives a more realistic check of whether the model generalises to unseen BLEVE setups.

## 12. References

- Python `difflib.get_close_matches`: https://docs.python.org/3/library/difflib.html#difflib.get_close_matches
- pandas `GroupBy`: https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.groupby.html
- scikit-learn `HistGradientBoostingRegressor`: https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.HistGradientBoostingRegressor.html
- scikit-learn `MLPRegressor`: https://scikit-learn.org/stable/modules/generated/sklearn.neural_network.MLPRegressor.html
- scikit-learn `PowerTransformer`: https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.PowerTransformer.html
- scikit-learn `RandomForestRegressor`: https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.RandomForestRegressor.html
- scikit-learn `Ridge`: https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.Ridge.html
- scikit-learn `RidgeCV`: https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.RidgeCV.html
- scikit-learn `RobustScaler`: https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.RobustScaler.html
- scikit-learn `StandardScaler`: https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.StandardScaler.html
- Widanage, Mohotti, Lee, Wijesooriya, and Meddage (2024), explainable machine learning models in blast load prediction: https://doi.org/10.1016/j.engstruct.2024.118271
